# CKD Model Demo Notebook
**Chronic Kidney Disease Prediction - Interactive Model Demonstration**

This notebook demonstrates how to:
1. Load trained CKD prediction models
2. Run inference on new patient data
3. Evaluate model performance with metrics and visualizations
4. Interpret predictions with SHAP explanations

**Models included:** Random Forest, XGBoost, Logistic Regression, SVM, Decision Tree

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
import shap

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Libraries loaded successfully")

## 1) Load Trained Models and Artifacts

Loading all trained models, preprocessor, and test data from the `models/` directory.

In [ ]:
# Define paths
PROJECT_ROOT = Path(".").resolve()
MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data"

# Load all trained models
models = {
    "Random Forest": joblib.load(MODEL_DIR / "ckd_random_forest_tuned.joblib"),
    "XGBoost": joblib.load(MODEL_DIR / "ckd_xgboost_tuned.joblib"),
    "Logistic Regression": joblib.load(MODEL_DIR / "ckd_logistic_regression_tuned.joblib"),
    "SVM": joblib.load(MODEL_DIR / "ckd_svm_tuned.joblib"),
    "Decision Tree": joblib.load(MODEL_DIR / "ckd_decision_tree_tuned.joblib"),
}

# Load preprocessor and data
preprocessor = joblib.load(MODEL_DIR / "preprocessor.joblib")
X_train = np.load(MODEL_DIR / "X_train_processed.npy")
X_test = np.load(MODEL_DIR / "X_test_processed.npy")
y_train = np.load(MODEL_DIR / "y_train.npy")
y_test = np.load(MODEL_DIR / "y_test.npy")
feature_names = pd.read_csv(MODEL_DIR / "feature_names.csv")["feature"].tolist()

print(f"✓ Loaded {len(models)} models")
print(f"✓ Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"✓ Test set: {X_test.shape[0]} samples")
print(f"✓ Class distribution - Train: {np.bincount(y_train)} | Test: {np.bincount(y_test)}")

## 2) Model Performance Evaluation

Evaluate all models on the test set with standard classification metrics.

In [ ]:
# Generate predictions for all models
results = []

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

# Create results dataframe
results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False)
results_df = results_df.round(4)

print("Model Performance Comparison:\n")
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
x = np.arange(len(results_df))
width = 0.15

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']

for i, metric in enumerate(metrics):
    offset = width * (i - 2)
    ax.bar(x + offset, results_df[metric], width, label=metric, color=colors[i], alpha=0.8)

ax.set_xlabel('Models', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison Across Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df["Model"], rotation=15, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3) Confusion Matrix Visualization

Detailed confusion matrices for the top 2 performing models.

In [ ]:
# Plot confusion matrices for top 2 models
top_models = results_df["Model"].head(2).tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, model_name in enumerate(top_models):
    model = models[model_name]
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    # Plot confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Not CKD', 'CKD'], yticklabels=['Not CKD', 'CKD'],
                cbar_kws={'label': 'Count'})
    
    axes[idx].set_title(f'{model_name} - Confusion Matrix', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('True Label', fontsize=11)
    axes[idx].set_xlabel('Predicted Label', fontsize=11)
    
    # Add metrics text
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    axes[idx].text(0.5, -0.15, f'Accuracy: {acc:.3f} | F1: {f1:.3f}',
                   ha='center', transform=axes[idx].transAxes, fontsize=10)

plt.tight_layout()
plt.show()

## 4) ROC Curves

Receiver Operating Characteristic curves showing the trade-off between sensitivity and specificity.

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 8))

for model_name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    
    plt.plot(fpr, tpr, linewidth=2, label=f'{model_name} (AUC = {auc_score:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('ROC Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5) Single Patient Inference Demo

Demonstrate how to make predictions for a new patient.

In [ ]:
# Example patient data (healthy patient profile)
healthy_patient = {
    "age": 45, "bp": 80, "sg": 1.020, "al": 0, "su": 0,
    "bgr": 120, "bu": 30, "sc": 1.0, "sod": 140, "pot": 4.5,
    "hemo": 14.5, "pcv": 44, "wc": 8000, "rc": 5.0,
    "rbc": "normal", "pc": "normal", "pcc": "notpresent", "ba": "notpresent"
}

# Example patient data (CKD patient profile)
ckd_patient = {
    "age": 65, "bp": 90, "sg": 1.010, "al": 4, "su": 3,
    "bgr": 180, "bu": 80, "sc": 4.5, "sod": 135, "pot": 5.5,
    "hemo": 9.5, "pcv": 30, "wc": 12000, "rc": 3.5,
    "rbc": "abnormal", "pc": "abnormal", "pcc": "present", "ba": "present"
}

def predict_patient(patient_data, model_name="Random Forest"):
    """Make prediction for a single patient."""
    # Convert to DataFrame
    patient_df = pd.DataFrame([patient_data])
    
    # Preprocess
    X_patient = preprocessor.transform(patient_df)
    
    # Predict
    model = models[model_name]
    prediction = model.predict(X_patient)[0]
    probability = model.predict_proba(X_patient)[0]
    
    result = {
        "prediction": "CKD" if prediction == 1 else "Not CKD",
        "confidence": probability[prediction] * 100,
        "ckd_probability": probability[1] * 100,
        "not_ckd_probability": probability[0] * 100
    }
    
    return result

# Test on both example patients
print("="*60)
print("HEALTHY PATIENT PREDICTION:")
print("="*60)
result_healthy = predict_patient(healthy_patient, "Random Forest")
for key, value in result_healthy.items():
    print(f"{key.replace('_', ' ').title()}: {value if isinstance(value, str) else f'{value:.2f}%'}")

print("\n" + "="*60)
print("CKD PATIENT PREDICTION:")
print("="*60)
result_ckd = predict_patient(ckd_patient, "Random Forest")
for key, value in result_ckd.items():
    print(f"{key.replace('_', ' ').title()}: {value if isinstance(value, str) else f'{value:.2f}%'}")

## 6) Feature Importance Analysis

Understanding which features drive model predictions.

In [ ]:
# Get feature importance from Random Forest
rf_model = models["Random Forest"]
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(20)

colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
plt.barh(range(len(top_features)), top_features['importance'], color=colors)
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance Score', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title('Top 20 Most Important Features - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

## 7) SHAP Explanations (Model Interpretability)

SHAP (SHapley Additive exPlanations) values show how each feature contributes to individual predictions.

In [ ]:
# Load precomputed SHAP values or compute for Random Forest
rf_shap_test = np.load(MODEL_DIR / "rf_shap_values_test.npy")
if rf_shap_test.ndim == 3:
    rf_shap_test = rf_shap_test[:, :, 1]  # Extract class 1 (CKD)

# Create explainer
rf_explainer = shap.TreeExplainer(models["Random Forest"])

# SHAP Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(rf_shap_test, pd.DataFrame(X_test, columns=feature_names), 
                  plot_type="dot", show=False, max_display=15)
plt.title('SHAP Feature Impact Summary - Random Forest', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (mean absolute impact)
plt.figure(figsize=(10, 8))
shap.summary_plot(rf_shap_test, pd.DataFrame(X_test, columns=feature_names), 
                  plot_type="bar", show=False, max_display=15)
plt.title('Mean Absolute SHAP Values - Random Forest', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 8) Single Prediction Explanation

Explaining a specific patient's prediction using SHAP waterfall plot.

In [ ]:
# Pick a CKD patient from test set for detailed explanation
ckd_indices = np.where(y_test == 1)[0]
sample_idx = ckd_indices[0]  # First CKD patient in test set

# Get prediction
rf_model = models["Random Forest"]
prediction = rf_model.predict(X_test[sample_idx:sample_idx+1])[0]
probability = rf_model.predict_proba(X_test[sample_idx:sample_idx+1])[0]

print(f"Patient #{sample_idx} in Test Set")
print(f"True Label: {'CKD' if y_test[sample_idx] == 1 else 'Not CKD'}")
print(f"Predicted: {'CKD' if prediction == 1 else 'Not CKD'}")
print(f"CKD Probability: {probability[1]*100:.1f}%")
print("\n" + "="*60)

# SHAP explanation for this patient
sample_shap = rf_explainer.shap_values(X_test[sample_idx:sample_idx+1])
if isinstance(sample_shap, list):
    sample_shap = sample_shap[1]  # Class 1 (CKD)

# Waterfall plot
shap.waterfall_plot(
    shap.Explanation(
        values=sample_shap[0],
        base_values=rf_explainer.expected_value if isinstance(rf_explainer.expected_value, float) else rf_explainer.expected_value[1],
        data=X_test[sample_idx],
        feature_names=feature_names
    ),
    max_display=15,
    show=True
)

## 9) Interactive Inference: Custom Patient Entry

Modify the values below to test predictions on your own patient data.

In [ ]:
# Create your own patient profile here
custom_patient = {
    # Numeric features
    "age": 55,           # Age in years
    "bp": 85,            # Blood pressure (mm/Hg)
    "sg": 1.015,         # Specific gravity
    "al": 2,             # Albumin (0-5)
    "su": 1,             # Sugar (0-5)
    "bgr": 150,          # Blood glucose random (mg/dL)
    "bu": 60,            # Blood urea (mg/dL)
    "sc": 2.5,           # Serum creatinine (mg/dL)
    "sod": 138,          # Sodium (mEq/L)
    "pot": 5.0,          # Potassium (mEq/L)
    "hemo": 11.5,        # Hemoglobin (g/dL)
    "pcv": 35,           # Packed cell volume (%)
    "wc": 10000,         # White blood cell count (cells/cumm)
    "rc": 4.2,           # Red blood cell count (millions/cumm)
    
    # Categorical features
    "rbc": "normal",     # Red blood cells: normal/abnormal
    "pc": "normal",      # Pus cell: normal/abnormal
    "pcc": "notpresent", # Pus cell clumps: present/notpresent
    "ba": "notpresent",  # Bacteria: present/notpresent
}

# Get predictions from all models
print("="*70)
print("CUSTOM PATIENT PREDICTION - ALL MODELS")
print("="*70)

for model_name in models.keys():
    result = predict_patient(custom_patient, model_name)
    print(f"\n{model_name}:")
    print(f"  Prediction: {result['prediction']}")
    print(f"  Confidence: {result['confidence']:.2f}%")
    print(f"  CKD Probability: {result['ckd_probability']:.2f}%")

## 10) Summary & Model Recommendation

Based on the evaluation results, here are the key findings:

In [ ]:
# Print summary statistics
print("="*70)
print("MODEL EVALUATION SUMMARY")
print("="*70)
print("\nBest Performing Models:")
print(results_df.head(3).to_string(index=False))

best_model = results_df.iloc[0]["Model"]
best_f1 = results_df.iloc[0]["F1-Score"]
best_auc = results_df.iloc[0]["ROC-AUC"]

print(f"\n{'='*70}")
print(f"RECOMMENDATION: {best_model}")
print(f"{'='*70}")
print(f"  • Best F1-Score: {best_f1:.4f}")
print(f"  • ROC-AUC Score: {best_auc:.4f}")
print(f"  • Balanced performance on both CKD and Non-CKD cases")
print(f"  • Provides interpretable feature importance and SHAP explanations")
print(f"\n{'='*70}")
print("✓ Demo notebook completed successfully!")
print("  You can now use these models for clinical decision support.")
print(f"{'='*70}")